# Contribution A — Distribution-Aware Active Annotation

Runs the region-level annotation study on **real PROB proposals**, on one NVIDIA T4,
in roughly four to five hours.

**What it measures.** For a fixed *region-level* annotation budget, which candidate
regions should the oracle label so that the most rare unknown objects are found? The
acquisition score is the proposal's equation (1),

```text
s(x) = alpha * uncertainty(x) + beta * novelty(x) + gamma * rarity(x) * coherence(x)**p
```

where `coherence` gates **only** the rarity term, so an isolated proposal keeps its
uncertainty and novelty but loses its rarity bonus.

> **Read [`docs/results.md`](../docs/results.md) before interpreting any output.** The
> coherence gate does **not** improve tail discovery on this pool, reproducibly, and a
> one-line `objectness x box scale` prior finds 1.88x more unknown objects than the
> full distribution-aware score. That is a result about a named hypothesis in the
> proposal, not a bug to be fixed by rerunning this notebook.

This notebook is a **thin driver**. All logic lives in the repository, the protocol
lives in `configs/contribution_a.yaml`, and the entrypoint is
`experiments/contribution_a.py`. Nothing here reimplements anything, so what you run
in Colab is what runs in CI.

## 1. Setup

Clones the two repositories and installs the package. Re-runnable: an existing
checkout is left alone.

In [ ]:
# @title Clone and install
DAOWOD_REPO = "https://github.com/gubiczam/distribution-aware-owod.git"
PROB_REPO = "https://github.com/gubiczam/PROB.git"

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content") if IN_COLAB else Path.cwd().parent

def sh(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)

DAOWOD = ROOT / "distribution-aware-owod"
PROB = ROOT / "PROB"
if IN_COLAB:
    if not DAOWOD.exists():
        sh(["git", "clone", DAOWOD_REPO, str(DAOWOD)])
    if not PROB.exists():
        sh(["git", "clone", PROB_REPO, str(PROB)])
    sh([sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{DAOWOD}[dev]"])
else:
    DAOWOD = Path.cwd().parent

os.chdir(DAOWOD)
sh([sys.executable, "-m", "pytest", "-q", "--no-header", "-x", "tests/test_memory.py"])
print("\nrepository:", DAOWOD)

## 2. Configuration

Edit **only** this cell. Everything that defines the protocol — pool sizes, budgets,
rounds, seeds, arms, severity axis — lives in `configs/contribution_a.yaml`, because a
change to any of those changes what the numbers mean and belongs in version control
rather than in a notebook cell.

Leave `RUN_MODE = "DEBUG"` for the first pass: it exercises every stage on about 410
images in a few minutes and needs no GPU. Its numbers are **not** reportable, and the
run says so. Switch to `FAST`, then `MAIN`, once it passes.

`MAINREVEALED` is the eleven-arm follow-up: the free informativeness-prior control and
the label-anchored distribution term beside the untouched baseline, sharing one pool,
one severity axis, one seed set and one budget grid.

In [ ]:
# @title Run settings
RUN_MODE = "DEBUG"           # DEBUG | FAST | MAIN | MAINREVEALED
DRIVE_ROOT = "/content/drive/MyDrive/DAOWOD"

DATA_ROOT = f"{DRIVE_ROOT}/data/OWOD"          # JPEGImages/, Annotations/, ImageSets/
CHECKPOINT = f"{DRIVE_ROOT}/checkpoints/MOWODB/t1.pth"
SPLIT_FILE = f"{DATA_ROOT}/ImageSets/OWDETR/owdetr_t1_train.txt"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/contribution_a_{RUN_MODE.lower()}"
CACHE_DIR = f"{DRIVE_ROOT}/cache/contribution_a"

# Reuse a frozen export instead of running the detector. See docs/artifacts.md.
EXISTING_EXPORT = ""

if "google.colab" in __import__("sys").modules:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Preflight

Refuses to start rather than failing three hours in. Checks the environment, the
dataset, the checkpoint digest and the split disjointness, then prints a deterministic
estimate of runtime, disk and memory.

If the projection exceeds the declared budget the run **stops**. It does not shrink the
evaluation pool to fit: that would change the protocol after it was declared, and every
reported denominator with it. Choose a smaller mode, or raise
`runtime_budget_hours` in the config deliberately.

In [ ]:
# @title Validate before spending GPU time
from daowod.config import load_modes, resolve_mode

for name, mode in sorted(load_modes("configs/contribution_a.yaml").items()):
    flag = "reportable" if mode.research_grade else "not reportable"
    print(f"{name:<14} {mode.total_images:>5} images  {len(mode.strategies)} arms  "
          f"{len(mode.imbalance_settings)} severities  {len(mode.seeds)} seeds  [{flag}]")

mode = resolve_mode(RUN_MODE)
print()
print(f"selected: {mode.name} — {mode.description}")

## 4. Run

One command. Every stage writes its result under `output_dir/state` and is skipped when
that file already exists, so a disconnected Colab session resumes rather than restarts.
The two stages that cost real time — detector inference and the per-severity study
matrix — are cached finely enough that a lost session costs minutes, not hours.

In [ ]:
# @title Run the study
import sys

argv = [
    "study",
    "--config", "configs/contribution_a.yaml",
    "--mode", RUN_MODE,
    "--data-root", DATA_ROOT,
    "--split", SPLIT_FILE,
    "--output", OUTPUT_DIR,
    "--cache", CACHE_DIR,
]
if EXISTING_EXPORT:
    argv += ["--existing-export", EXISTING_EXPORT]
else:
    argv += ["--checkpoint", CHECKPOINT]
if RUN_MODE == "DEBUG":
    argv += ["--no-gpu"]

sys.argv = ["contribution_a.py", *argv]
exec(open("experiments/contribution_a.py").read(), {"__name__": "__main__", "__file__": "experiments/contribution_a.py"})

## 5. Results

The run writes one CSV per logical table, JSON manifests recording the configuration,
environment, splits and leakage verdict, the publication figures, and a markdown
research summary written from the numbers rather than asserted.

`selected_proposals.csv` is the hand-off point for a downstream retraining experiment:
it is the annotation set the campaign actually bought.

In [ ]:
# @title List and preview the artifacts
from pathlib import Path
import pandas as pd

output = Path(OUTPUT_DIR)
for path in sorted(output.glob("*")):
    if path.is_file():
        print(f"{path.stat().st_size/1024:>9.1f} KB  {path.name}")

summary = output / "research_summary.md"
if summary.exists():
    from IPython.display import Markdown, display
    display(Markdown(summary.read_text()))

curves = output / "budget_curves_aggregated.csv"
if curves.exists():
    display(pd.read_csv(curves).head(20))

## 6. What this notebook does not do

* It does **not** retrain PROB, so it claims no `known mAP`, `U-Recall`, `WI` or `A-OSE`.
  The official PROB evaluator remains the only source for those.
* It does **not** measure catastrophic forgetting, which needs real incremental updates.
* It does **not** run Contribution B. The exemplar-allocation core is
  `src/daowod/memory.py` with its own entrypoint,
  `experiments/contribution_b.py`; measuring the optimal `alpha` is a separate future
  step documented in [`docs/research_design.md`](../docs/research_design.md) section 8.